# Statistical Consistency Verification

## Overview
This notebook documents p-value discrepancies > 0.05 found in public A/B test summaries, providing justifications per Constitution Principle VI (Transparency and Reproducibility).

## Reference
- **Paper**: [Statistical Consistency in A/B Testing](https://arxiv.org/abs/1905.08726)
- **Constitution Principle VI**: All statistical claims must be transparent, reproducible, and accompanied by sufficient methodological detail.

In [ ]:
import json
import pandas as pd
from pathlib import Path
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

PROJECT_ROOT = Path(".").resolve()
AUDIT_REPORT_PATH = PROJECT_ROOT / "output" / "audit_report.json"
DISCREPANCY_REPORT_PATH = PROJECT_ROOT / "output" / "p_value_discrepancies.csv"

## Load Audit Report
Read the audit report generated by the pipeline (T025).

In [ ]:
def load_audit_report():
    if not AUDIT_REPORT_PATH.exists():
        raise FileNotFoundError(f"Audit report not found at {AUDIT_REPORT_PATH}. "
                                "Please run the pipeline (T032) first.")
    
    with open(AUDIT_REPORT_PATH, 'r') as f:
        data = json.load(f)
    
    # The report structure depends on T025 output format
    # Assuming 'records' or direct list
    records = data.get('records', data) if isinstance(data, dict) else data
    
    logger.info(f"Loaded {len(records)} audit records from {AUDIT_REPORT_PATH}")
    return records

audit_records = load_audit_report()

## Identify P-Value Discrepancies > 0.05
Filter records where the absolute difference between reported and reconstructed p-values exceeds 0.05.

In [ ]:
def extract_discrepancies(records, threshold=0.05):
    discrepancies = []
    
    for record in records:
        reported_p = record.get('reported_p_value')
        reconstructed_p = record.get('reconstructed_p_value')
        
        if reported_p is None or reconstructed_p is None:
            continue
        
        diff = abs(float(reported_p) - float(reconstructed_p))
        
        if diff > threshold:
            discrepancies.append({
                'url': record.get('url', 'N/A'),
                'domain': record.get('domain', 'N/A'),
                'reported_p_value': reported_p,
                'reconstructed_p_value': reconstructed_p,
                'absolute_difference': diff,
                'sample_size': record.get('sample_size', 'N/A'),
                'effect_size': record.get('effect_size', 'N/A'),
                'test_type': record.get('test_type', 'N/A'),
                'inconsistency_flag': record.get('is_inconsistent', False),
                'notes': record.get('notes', '')
            })
    
    logger.info(f"Found {len(discrepancies)} records with p-value discrepancy > {threshold}")
    return discrepancies

discrepancies = extract_discrepancies(audit_records)

## Generate Justifications per Constitution Principle VI
For each discrepancy, provide a methodological justification.

In [ ]:
def generate_justifications(discrepancies):
    justified_records = []
    
    for rec in discrepancies:
        justification = []
        
        # Principle VI: Transparency and Reproducibility
        justification.append("**Constitution Principle VI Justification**:")
        
        # Analyze potential causes
        if rec['test_type'] == 'binary':
            justification.append("- **Test Type**: Binary outcome (proportions).")
            justification.append("- **Potential Cause**: Discrepancy may stem from use of Fisher's Exact Test vs. Z-test approximation, or continuity correction differences.")
        elif rec['test_type'] == 'continuous':
            justification.append("- **Test Type**: Continuous outcome (means).")
            justification.append("- **Potential Cause**: Discrepancy may arise from unequal variance assumptions (Welch's t-test vs. Student's t-test) or non-normality.")
        else:
            justification.append(f"- **Test Type**: {rec['test_type'] or 'Unknown'}.")
        
        # Sample size check
        if rec['sample_size'] != 'N/A':
            try:
                n = int(rec['sample_size'])
                if n < 30:
                    justification.append("- **Sample Size Concern**: Small sample size (<30) may lead to instability in p-value estimation and approximation errors.")
            except (ValueError, TypeError):
                justification.append("- **Sample Size**: Missing or invalid.")
        
        # Effect size check
        if rec['effect_size'] != 'N/A':
            justification.append(f"- **Effect Size**: {rec['effect_size']}.")
        
        # Specific discrepancy note
        justification.append(f"- **Observed Difference**: |{rec['reported_p_value']:.4f} - {rec['reconstructed_p_value']:.4f}| = {rec['absolute_difference']:.4f}.")
        justification.append("- **Action Required**: Review original methodology, check for data transformation, or request raw data for re-analysis.")
        
        rec['justification'] = "\n".join(justification)
        justified_records.append(rec)
    
    return justified_records

justified_discrepancies = generate_justifications(discrepancies)

## Export Discrepancy Report
Write the justified discrepancies to a CSV file for pipeline acceptance.

In [ ]:
def write_discrepancy_report(records, output_path):
    if not records:
        logger.warning("No discrepancies found. Writing empty report.")
        # Create empty file with headers
        pd.DataFrame(columns=['url', 'domain', 'reported_p_value', 'reconstructed_p_value', 
                             'absolute_difference', 'sample_size', 'effect_size', 
                             'test_type', 'inconsistency_flag', 'notes', 'justification']).to_csv(output_path, index=False)
    else:
        df = pd.DataFrame(records)
        df.to_csv(output_path, index=False)
    
    logger.info(f"Discrepancy report written to {output_path}")

write_discrepancy_report(justified_discrepancies, DISCREPANCY_REPORT_PATH)

## Summary Statistics
Display summary of discrepancies found.

In [ ]:
if justified_discrepancies:
    print(f"Total Discrepancies Found: {len(justified_discrepancies)}")
    print("\nSample of Discrepancies (first 5):")
    for i, rec in enumerate(justified_discrepancies[:5]):
        print(f"\n--- Record {i+1} ---")
        print(f"URL: {rec['url']}")
        print(f"Domain: {rec['domain']}")
        print(f"Reported P: {rec['reported_p_value']}")
        print(f"Reconstructed P: {rec['reconstructed_p_value']}")
        print(f"Difference: {rec['absolute_difference']:.4f}")
        print(f"\nJustification:\n{rec['justification']}")
else:
    print("No p-value discrepancies > 0.05 found in the audit report.")
    print("This indicates high statistical consistency in the analyzed corpus.")